In [16]:
import panel as pn

In [17]:
pn.extension()

In [18]:
from pinotdb import connect
import pandas as pd

conn = connect(host='localhost', port=8099, path='/query/sql', scheme='http')

list_of_samples = []

def get_changes():
    query = """
        select 
                count(*) AS events1Min,
                distinctcount(user) AS users1Min,
                distinctcount(domain) AS domains1Min
        from wikievents_REALTIME
        where ts > ago('PT1M')
        limit 1;
    """

    curs = conn.cursor()

    curs.execute(query)
    
    temp_df = pd.DataFrame(curs, columns=[item[0] for item in curs.description])
    temp_df['sample_time'] = pd.Timestamp.now()

    list_of_samples.append(temp_df)
    if len(list_of_samples) > 30:
        list_of_samples.pop(0)

    return temp_df.to_dict('records')[0], pd.concat(list_of_samples).sort_values(by=["sample_time"])

In [19]:
get_changes()

({'events1Min': 2263,
  'users1Min': 361,
  'domains1Min': 82,
  'sample_time': Timestamp('2026-05-12 12:29:10.846823')},
    events1Min  users1Min  domains1Min                sample_time
 0        2263        361           82 2026-05-12 12:29:10.846823)

In [20]:
# Necessary for reactive pandas
import hvplot.pandas 

sample_df, samples_df = get_changes()
table_changes = pn.rx(sample_df)
samples_df_rx = pn.rx(samples_df)

## Extract Data

def update_table_changes():
    sample_df, samples_df = get_changes()
    table_changes.rx.value = sample_df
    samples_df_rx.rx.value = samples_df

pn.state.add_periodic_callback(update_table_changes, period=60000)

PeriodicCallback(callback=<function update_table_changes at 0x0000020F641A9C70>, count=None, counter=0, log=True, name='PeriodicCallback07717', period=60000, running=True, session_scoped=True, timeout=None)

In [21]:
sample_configs = {
    "events1Min": {"label": "Current 1min Events Sample", "metric": "events"},
    "users1Min": {"label": "Current 1min Users Sample", "metric": "users"},
    "domains1Min": {"label": "Current 1min Domains Sample", "metric": "domains"},
}

observed_variables = {
    sample_name: table_changes.rx()[sample_name] for sample_name, _ in sample_configs.items()
}

In [22]:
samples_views = {
    sample_name: pn.indicators.Number(
        name=sample_config["label"],
        value=observed_variables[sample_name],
        format="{value} " + f"{sample_config['metric']}/minute"
    )
    for sample_name, sample_config in sample_configs.items()
}

In [23]:
metric_indicator = pn.Column(
    "1 Minute Changes",
    pn.Row(samples_views["events1Min"]),
    pn.Row(samples_views["users1Min"]),
    pn.Row(samples_views["domains1Min"]),
)

# metric_indicator.servable()

In [24]:
fig_events = samples_df_rx.hvplot(kind="line", x="sample_time", y="events1Min", title="Plot Events")
plot_events = pn.pane.HoloViews(fig_events, name="Plot Events")
fig_users = samples_df_rx.hvplot(kind="line", x="sample_time", y="users1Min", title="Plot Users")
plot_users = pn.pane.HoloViews(fig_users, name="Plot Users")
fig_domains = samples_df_rx.hvplot(kind="line", x="sample_time", y="domains1Min", title="Plot Domains")
plot_domains = pn.pane.HoloViews(fig_domains, name="Plot Domains")

tabs = pn.Tabs(
    plot_events,
    plot_users,
    plot_domains,
    sizing_mode="stretch_width",
    height=500,
    margin=10,
)

In [25]:
pn.Column(metric_indicator, tabs).servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'141ef5e5-cffd-4173-b436-0986059bdf6a': {'version…

UnknownReferenceError: can't resolve reference 'feaf454a-3082-4059-8e4f-6192e3be2aca'

UnknownReferenceError: can't resolve reference 'feaf454a-3082-4059-8e4f-6192e3be2aca'